        АНАЛИЗ ИНЦИДЕНТОВ ДАТА-КАЧЕСТВА (DQ INCIDENTS ANALYSIS)

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd

    Вспомогательные функции для визуализации и анализа

In [ ]:
# Настройка графиков.
plt.style.use('seaborn-v0_8' if 'seaborn-v0_8' in plt.style.available else 'default')

def plots_ABS_and_Normal(data_absl, data_norm, str_name, tuple_n=(14, 5)):
    """Строит два графика: абсолютное и нормализованное распределения."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=tuple_n)

    # Абсолютное распределение
    ax1.bar(data_absl.index, data_absl.values, edgecolor='black', alpha=0.7)
    ax1.set_xlabel(f'Уровень {str_name}')
    ax1.set_ylabel('Количество')
    ax1.set_title(f'Распределение {str_name} (Абсолютное)')
    ax1.grid(axis='y', linestyle='--', alpha=0.5)

    # Нормализованное распределение
    ax2.bar(data_norm.index, data_norm.values, edgecolor='black', alpha=0.7)
    ax2.set_xlabel(f'Уровень {str_name}')
    ax2.set_ylabel('Доля')
    ax2.set_title(f'Распределение {str_name} (Нормализованное)')
    ax2.grid(axis='y', linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.show()


def plot_pivot(pivot_table, str_x, str_y):
    """Визуализация сводной таблицы в виде столбчатой диаграммы."""
    pivot_table.plot(kind='bar', figsize=(12, 6))
    plt.title(f'Распределение {str_y} по {str_x}')
    plt.xlabel(f'{str_x}')
    plt.ylabel('Количество')
    plt.legend(title=f'{str_y}', loc='upper right')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


def plot_Time_Row(table_pivot, date_name, value_name):
    """Визуализация динамики количества инцидентов по датам."""
    fig, ax = plt.subplots(figsize=(14, 7))
    ax.plot(
        table_pivot[date_name],
        table_pivot[value_name],
        marker='o',
        linewidth=2,
        markersize=4,
        color='blue',
        linestyle='-',
    )

    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%m-%Y'))
    ax.xaxis.set_major_locator(mdates.DayLocator(interval=14))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_title('Динамика количества инцидентов по дням', fontsize=16, pad=20)
    ax.set_xlabel('Дата', fontsize=12)
    ax.set_ylabel('Количество инцидентов', fontsize=12)

    plt.tight_layout()
    plt.show()

    1. Загрузка данных

In [ ]:
df_incidents = pd.read_csv(r'dq_incidents.csv')
df_checks = pd.read_csv(r'data_quality_checks_log.csv')
df_model = pd.read_csv(r"model_monitoring.csv")

    2. Пропуски и замена типов данных.

In [ ]:
df_incidents['resolution_time_hours'] = df_incidents['resolution_time_hours'].fillna(df_incidents['resolution_time_hours'].mean())
df_checks['actual_value'] = df_checks['actual_value'].fillna(df_checks['actual_value'].mean())

df_incidents['incident_date'] = pd.to_datetime(df_incidents['incident_date'])
df_checks['execution_datetime '] = pd.to_datetime(df_checks['execution_datetime'])
df_model['monitoring_date'] = pd.to_datetime(df_model['monitoring_date'])
df_model['monitoring_time'] = pd.to_datetime(df_model['monitoring_time'])

    3. Анализ распределений (Severity, Status, Data Source)

In [ ]:
data_severity_absolute = df_incidents['severity'].value_counts()
data_severity_normir = df_incidents['severity'].value_counts(normalize=True)

data_status_absolute = df_incidents['status'].value_counts()
data_status_normir = df_incidents['status'].value_counts(normalize=True)

data_source_absolute = df_incidents['data_source'].value_counts()
data_source_normir = df_incidents['data_source'].value_counts(normalize=True)

# Построение графиков распределений
plots_ABS_and_Normal(data_absl=data_severity_absolute, data_norm=data_severity_normir, str_name="Severity", tuple_n=(17, 5))
plots_ABS_and_Normal(data_absl=data_status_absolute, data_norm=data_status_normir, str_name="Status", tuple_n=(17, 5))
plots_ABS_and_Normal(data_absl=data_source_absolute, data_norm=data_source_normir, str_name="Data_Source", tuple_n=(25, 5))

    4. Сводные таблицы и кросс-анализ

In [ ]:
table_pivot_source_severity = pd.crosstab(df_incidents['data_source'], df_incidents['severity'])
plot_pivot(pivot_table=table_pivot_source_severity, str_x=table_pivot_source_severity.index.name, str_y=table_pivot_source_severity.columns.name)

    5. Динамика инцидентов по датам

In [ ]:
daily_incidents = df_incidents.groupby('incident_date').size().reset_index(name='count')
print("Общее число уникальных дат:", len(daily_incidents))
plot_Time_Row(daily_incidents, date_name='incident_date', value_name='count')

    6. Статистика по инцидентам. 

In [ ]:
total_quantity_incidents = df_incidents['incident_id'].nunique() # 1.

mean_time_of_resolution = (df_incidents['resolution_time_hours'].mean()).round(2) # 3.

SLA_breach_rate = (df_incidents['sla_breach'].mean()*100).round(2)  # 4.

distribution_of_reccurent = (df_incidents['is_recurrent'].mean() * 100).round(2) # 5.

top_root_causes = (df_incidents['root_cause'].value_counts(normalize= True)*100).round(3) # Список root cases с частотностью.

matrix_of_correlation = (df_incidents[['affected_records', 'total_records', 'resolution_time_hours', 'error_rate_percent']]
                         .corr()) # 8.

mean_error_rate_by_check_type = (df_incidents.groupby('check_type')['error_rate_percent'] # 7.
                                 .mean().round(2).reset_index().
                                 sort_values(by= 'error_rate_percent', ascending= False))


print(f"Общее количество инцидентов: {total_quantity_incidents}" '\n\n'
      f"Среднее время разрешения инцидента: {mean_time_of_resolution} часов", '\n\n'
      f"Процент нарушенных периодов решения: {SLA_breach_rate} %", '\n\n'
      f"Процент повторно открывающихся инцидентов: {distribution_of_reccurent} %", '\n\n',
      f"Наиболее популярные причины ошибок (в процентах): \n{top_root_causes.head(5)}", '\n\n',
      "Средний процент ошибки по типам проверок: \n", mean_error_rate_by_check_type)